# Part 2 — External Database Configuration

*Last updated:* 2026-11-12

This notebook is the step-by-step guide for creating and validating the **external YAML** files that control which **external databases**
(HGNC, MGI, UniProt, RefSeq, …) are included when IDTrack builds the identifier graph.

By the end, you will:
- have `<organism>_externals_modified.yml` files in your local repository
- understand why keeping your external database selection **small and curated** improves mapping quality
- know how to validate that your YAML is compatible with your chosen snapshot boundary

This notebook covers:
- **2.1 Human** (*Homo sapiens*) — multi-assembly (GRCh38 + GRCh37, and older archives when available)
- **2.2 Mouse** (*Mus musculus*) — clean handoff (one maintained assembly per release: NCBIM37 → GRCm38 → GRCm39)
- **2.3 Pig** (*Sus scrofa*) — clean handoff (one maintained assembly per release: Sscrofa9 → Sscrofa10.2 → Sscrofa11.1)
- **2.4 Rhesus macaque** (*Macaca mulatta*) — clean handoff (MMUL_1 → Mmul_8.0.1 → Mmul_10)
- **2.5 Cynomolgus macaque** (*Macaca fascicularis*) — clean handoff (Macaca_fascicularis_5.0 → 6.0)
- **2.6 Extending a YAML to a new Ensembl release** (without regenerating it)
- **2.7 Adding a new organism** (advanced; one table entry in `idtrack/organisms.py`, or `idtrack.organisms.register_organism` for a session)

IDTrack ships a default `<organism>_externals_modified.yml` for all five organisms (in `idtrack/default_config/`), made exactly
as this notebook shows: a template from every release and form, then `Include: true` for an allowlist. You only need your own
file to change the selection (a file in your local repository takes precedence over the shipped one) or for a new organism.

In IDTrack, assemblies are a first-class dimension (not just “primary vs legacy”): the templates keep all assembly entries that Ensembl
exposes for the species. Keeping them is usually the right choice, especially when you integrate datasets annotated with different GTFs or
reference packages.

> **Tip:** If you're new, start with `00_idtrack_overview.ipynb` (concepts) and `01_installation_guide.ipynb` (setup).

> **This notebook is shown without outputs.** Its central cells read Ensembl's tables of every release of an organism to make the template (`create_database_content()`): from tables already downloaded this takes about 2.5 hours for human, 45 minutes for mouse and 5 to 10 minutes for pig and the macaques (one process), and the first time it also downloads them; every cell that applies an allowlist needs that template. It was last executed (without those cells) with the development code of 12 November 2026; the allowlists have changed since (NCBI's gene ids are selected on every form), so its outputs no longer matched this version and were removed. The externals files shipped with this version were made again with its code from Ensembl's tables up to release 115 and came out the same: the templates entry for entry, the modified files as the templates with the allowlists below applied (which the unit tests check byte for byte).

## 2.0 — Pre-requisites (what you need before you start)

- A working Python environment with IDTrack installed (`pip install "git+https://github.com/theislab/idtrack.git"`)
- Network access (first-time runs download Ensembl metadata)
- A writable **local repository** folder (IDTrack cache)

**What you get at the end** (one per organism you work through):
- `homo_sapiens_externals_modified.yml`
- `mus_musculus_externals_modified.yml`
- `sus_scrofa_externals_modified.yml`
- `macaca_mulatta_externals_modified.yml`
- `macaca_fascicularis_externals_modified.yml`

Each file lives in your local repository and is safe to share with collaborators. Without them, IDTrack uses the shipped defaults.

In [ ]:
# Load notebook utilities (collapsible output magic for tutorials). `_notebook_utils.py` sits next to the notebooks in
# IDTrack's repository; a notebook downloaded on its own runs without it and shows the logs as they come.
try:
    get_ipython().run_line_magic('load_ext', '_notebook_utils')
except ModuleNotFoundError:
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def collapse(line, cell):
        get_ipython().run_cell(cell)

In [ ]:
# 1) Setup (run this once)
from __future__ import annotations

import os
from pathlib import Path

import yaml
import idtrack

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

from idtrack import API

API.configure_logger()

print('Local repository:', LOCAL_REPOSITORY)

### What is this local repository?

Think of it as your **IDTrack workspace**. It will contain:
- cached Ensembl tables (downloaded once, reused many times)
- graph snapshot files (`graph_<organism>_assembly<key>_min<first>_max<snapshot>.npz`)
- your external YAML files (`*_externals_modified.yml`)

If you work on multiple projects, you can use one shared local repository (big but convenient),
or separate local repositories (clean separation, easier to archive/share).


## 2. External YAML in plain language

The YAML answers the question:

> **Which external databases should IDTrack trust and include as edges in the graph?**

Ensembl knows about *many* external resources. Some are high quality and useful, some are redundant, and some
create a lot of branching (ambiguity).

A good rule of thumb:
- enable a **small, curated** set of strong databases
- avoid enabling lots of near-duplicates

### YAML structure (what you will see)

The generated template is nested like this:

```yaml
<organism>:
  gene:
    <Database Name>:
      Assembly:
        <assembly_code>:
          Ensembl release: "90,91,92,..."
          Include: false
```

**You mainly edit one thing:** `Include: false` → `Include: true` (a YAML boolean, without quotes; IDTrack refuses anything else and
names the entry). Entries for an assembly the organism does not have, or for releases outside the assembly's range in IDTrack's
organism table, are left out with a warning.

Why is there an `Assembly:` level?

- Some external databases are only available (or only well-populated) on specific assembly/release combinations.
- IDTrack reads an external database for the assembly and releases an entry enables, so it needs to know which edges exist in which
  build context.

For most curated databases, enabling them for **all assemblies listed in the template** is the right default.


## 3. Choose your snapshot release (reproducibility knob)

When you build graphs later, you will pick a **snapshot release** (maximum Ensembl release).

For beginners, a good choice is a recent release that the shipped YAML files cover (they cover up to release 116); the tutorials use 115.
Pin it in your scripts: a graph is built per snapshot release, and the newest Ensembl release changes every few months.

For reproducible research projects, consider pinning:
- the Ensembl release used by your reference annotation (e.g. the one your pipeline used)
- or the release used by a published dataset you integrate


## 2.1 — Human (Homo sapiens)

IDTrack ships a **default external YAML** for human (as for every supported organism). Human is special in another way:
GRCh38 and GRCh37 overlap for many releases, so two assemblies are in play at once.

An important conceptual point before you start editing:

- Assemblies are not just “primary vs legacy” in IDTrack.
- Assemblies are part of the mapping problem.

This matters in real projects (especially atlas building), where you often combine datasets annotated with different genome builds:
- GRCh37-era references / GTFs
- GRCh38-era references / GTFs

IDTrack’s snapshot graphs are **multi-assembly**. Keeping the assembly blocks of a database enabled in the external YAML means:
- identifiers of that database are recognised at the releases of every assembly (GRCh37-era inputs need the `37` entries)
- a conversion to a release of an earlier assembly can return that assembly's identifiers (the final conversion is made on the assembly
  Ensembl maintained at the target release)

For human, the shipped default YAML already includes the common human assembly codes exposed by Ensembl (typically `38` = GRCh38 and
`37` = GRCh37). Depending on release coverage, you may also see older archive assemblies in templates.

You have two good options:

1. **Use the shipped default as a starting point** (fast, recommended for most users)
2. **Regenerate a fresh template from live Ensembl metadata** (slower, but useful if you want to refresh external database lists)

Recommended external databases for human (high signal, widely used), with the form whose section of the YAML holds them:
- **HGNC Symbol** (gene; human-readable gene symbols)
- **EntrezGene** (gene)
- **Uniprot/SWISSPROT** (translation; reviewed UniProt accessions)
- **RefSeq_mRNA** (transcript; optionally also `RefSeq_peptide`, translation, if your workflow needs proteins)

> **Warning:** Assembly codes are species-specific. `38` means GRCh38 for human, but `38` means GRCm38 for mouse.

> **Tip:** Keep your allowlist small at first. You can expand later once you understand how ambiguity shows up in your results.


In [ ]:
# 4.1 Resolve organism name + latest release
organism_hs, latest_release_hs = API.ensembl_organism('human')
SNAPSHOT_RELEASE_HS = 115  # pinned (latest_release_hs is Ensembl's newest release)
organism_hs, SNAPSHOT_RELEASE_HS

### 2.1.1 Create a DatabaseManager snapshot

This object is responsible for talking to Ensembl (downloads + caching) **bounded by your snapshot release**.


In [ ]:
dm_hs = API.database_manager(organism_hs, SNAPSHOT_RELEASE_HS, str(LOCAL_REPOSITORY))
dm_hs

### 2.1.2 Generate a template YAML (optional: every supported organism ships a default)

This step can take time, because IDTrack enumerates database metadata across releases.
You typically do it once per organism (and then keep the YAML).


In [ ]:
%%collapse Click to show download logs
# Included for tutorial purposes only.

# NOTE: This can take a while on first run (network + caching).
df_hs = dm_hs.create_database_content()
dm_hs.external_inst.create_template_yaml(df_hs)

template_hs = Path(dm_hs.external_inst.file_name_template_yaml())
modified_hs = Path(dm_hs.external_inst.file_name_modified_yaml(location='configured'))
{
    'template_yaml': str(template_hs),
    'modified_yaml': str(modified_hs),
}

### 2.1.3 Edit the YAML (two options)

#### Option A — edit by hand (recommended at least once)
1. Open the template file shown above (ends with `_externals_template.yml`).
2. Search for a database you care about (e.g. `HGNC Symbol`).
3. Change `Include: false` to `Include: true` for **every assembly block listed** (recommended).
   - This keeps the database available as a bridge regardless of which genome build your inputs were annotated against.
   - Only restrict to a subset of assemblies if you intentionally want a build-specific configuration.
4. Save as `_externals_modified.yml` in the same folder (IDTrack will look for this first).

#### Option B — programmatic toggles (great for reproducibility)
The next cell applies the **shipped default configuration** used by IDTrack (`homo_sapiens_externals_modified.yml`).

This default enables **63 form/database/assembly combinations** across three forms:
- **gene** (35 combinations): HGNC Symbol, EntrezGene, NCBI gene, UniProtKB Gene Name, Havana gene, Vega gene, RFAM, Clone-based identifiers, and their synonym variants
- **transcript** (14 combinations): CCDS, Havana transcript, RefSeq mRNA/ncRNA (curated + predicted)
- **translation** (14 combinations): Havana translation, RefSeq peptide (curated + predicted), UniProt (Swiss-Prot + TrEMBL)

You can modify `DEFAULT_HS_EXTERNALS` in the cell below to customize your allowlist. The allowlists of this notebook are the ones
the shipped files were made from; the package keeps the same lists in `idtrack.build.default_externals.DEFAULT_EXTERNALS` together with
`apply_allowlist(template, organism, allowlist)`, which does what the loop below does.

In [ ]:
# Default external databases for HUMAN (matches the shipped default config)
# This dict is structured as: form -> database -> list of assemblies where Include=True
# These are the defaults IDTrack ships for homo_sapiens (idtrack.build.default_externals.DEFAULT_EXTERNALS).

DEFAULT_HS_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [36, 37, 38],
        'Clone_based_vega_gene': [36, 37, 38],
        'EntrezGene': [36, 37, 38],
        'HGNC Symbol': [36, 37, 38],
        'Havana gene': [36, 37, 38],
        'NCBI gene (formerly Entrezgene)': [36, 37, 38],
        'RFAM': [36, 37, 38],
        'RFAM label': [36, 37, 38],
        'miRBase': [36, 37, 38],
        'miRBase label': [36, 37, 38],
        'UniProtKB Gene Name': [36, 37, 38],
        'Vega gene': [36, 37, 38],
        'Vega_gene': [36, 37, 38],
        'synonym_id::HGNC Symbol': [36, 37, 38],
        'synonym_id::NCBI gene (formerly Entrezgene)': [36, 37, 38],
        'synonym_id::UniProtKB Gene Name': [36, 37, 38],
    },
    'transcript': {
        'CCDS': [36, 37, 38],
        'CCDS label': [36, 37, 38],
        'Havana transcript': [36, 37, 38],
        'RefSeq_mRNA': [36, 37, 38],
        'RefSeq_mRNA_predicted': [36, 37, 38],
        'RefSeq_ncRNA': [36, 37, 38],
        'RefSeq_ncRNA_predicted': [36, 37, 38],
        'EntrezGene': [36, 37, 38],
        'NCBI gene (formerly Entrezgene)': [36, 37, 38],
        'synonym_id::NCBI gene (formerly Entrezgene)': [36, 37, 38],
    },
    'translation': {
        'Havana translation': [36, 37, 38],
        'RefSeq_peptide': [36, 37, 38],
        'RefSeq_peptide_predicted': [36, 37, 38],
        'Uniprot/SPTREMBL': [36, 37, 38],
        'Uniprot/SWISSPROT': [36, 37, 38],
        'EntrezGene': [36, 37, 38],
        'NCBI gene (formerly Entrezgene)': [36, 37, 38],
        'synonym_id::NCBI gene (formerly Entrezgene)': [36, 37, 38],
    },
}

# Load the template YAML
y = yaml.safe_load(template_hs.read_text(encoding='utf-8'))

# Apply the default configuration: set Include=True for matching form/database/assembly
enabled_count = 0
for form, databases in DEFAULT_HS_EXTERNALS.items():
    if form not in y[organism_hs]:
        continue
    for db_name, assemblies in databases.items():
        if db_name not in y[organism_hs][form]:
            print(f'  [SKIP] {form}/{db_name} not found in template')
            continue
        for asm_code in assemblies:
            asm_str = str(asm_code)
            if asm_str in y[organism_hs][form][db_name]['Assembly']:
                y[organism_hs][form][db_name]['Assembly'][asm_str]['Include'] = True
                enabled_count += 1

modified_hs.write_text(yaml.safe_dump(y, sort_keys=False, allow_unicode=True), encoding='utf-8')

print(f'Enabled {enabled_count} form/database/assembly combinations (the shipped default).')
print('Wrote:', modified_hs)

### 2.1.4 Validate + preview your selections

This load step is important: it confirms your YAML contains your snapshot release and that IDTrack can parse it.


In [ ]:
_ = dm_hs.external_inst.load_modified_yaml()
print('Enabled DBs:', dm_hs.external_inst.give_list_for_case('db'))
print('Assemblies in play:', dm_hs.external_inst.give_list_for_case('assembly'))

# dm_hs was created with snapshot_release=115; NCBI36 (assembly 36) serves human releases 48 to 54 only (GRCh37 from 55),
# so the NCBI36 entries are seen at an early release.
_dm_hs = dm_hs.change_release_auto_assembly(50)
_ = _dm_hs.external_inst.load_modified_yaml()
print('Enabled DBs:', _dm_hs.external_inst.give_list_for_case('db'))
print('Assemblies in play:', _dm_hs.external_inst.give_list_for_case('assembly'))

## 2.2 — Mouse (Mus musculus)

Mouse ships with a default external YAML too; the workflow below regenerates it (or your own variant):

1. generate a template YAML (from Ensembl metadata)
2. enable a curated allowlist of databases
3. validate the YAML


Mouse templates list multiple assemblies across history, but Ensembl is a clean-handoff species (one maintained assembly per release). You will usually see assembly codes like:
- `39` = GRCm39
- `38` = GRCm38
- `37` = NCBIM37

> **Tip:** For clean-handoff species, enabling the same database across the listed assemblies is usually fine;
> the per-assembly release ranges are disjoint, so you typically do not get overlapping assemblies within a single release.

Mouse-specific, commonly useful databases:
- **MGI Symbol** (mouse gene symbols)
- **EntrezGene**
- **Uniprot/SWISSPROT** (translation form)
- **RefSeq_mRNA** (transcript form)

> **Expected output:** You will create `mus_musculus_externals_modified.yml` in your local repository.


In [ ]:
# 5.1 Resolve organism name + latest release
organism_mm, latest_release_mm = API.ensembl_organism('mus musculus')
SNAPSHOT_RELEASE_MM = 115
organism_mm, SNAPSHOT_RELEASE_MM

In [ ]:
# 5.2 Create DatabaseManager snapshot
dm_mm = API.database_manager(organism_mm, SNAPSHOT_RELEASE_MM, str(LOCAL_REPOSITORY))
dm_mm

In [ ]:
%%collapse Click to show download logs
# Included for tutorial purposes only.

# 5.3 Generate template YAML
df_mm = dm_mm.create_database_content()
dm_mm.external_inst.create_template_yaml(df_mm)

template_mm = Path(dm_mm.external_inst.file_name_template_yaml())
modified_mm = Path(dm_mm.external_inst.file_name_modified_yaml(location='configured'))
{
    'template_yaml': str(template_mm),
    'modified_yaml': str(modified_mm),
}

### 2.2.1 Programmatic allowlist (mouse)

Mouse gene symbols come from **MGI** (Mouse Genome Informatics), so a typical set includes `MGI Symbol`.


In [ ]:
# Default external databases for MOUSE.
# This dict is structured as: form -> database -> list of assemblies where Include=True

DEFAULT_MM_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [37, 38, 39],
        'Clone_based_vega_gene': [37, 38, 39],
        'EntrezGene': [37, 38, 39],
        'MGI Symbol': [37, 38, 39],
        'Havana gene': [37, 38, 39],
        'NCBI gene (formerly Entrezgene)': [37, 38, 39],
        'RFAM': [37, 38, 39],
        'RFAM label': [37, 38, 39],
        'miRBase': [37, 38, 39],
        'miRBase label': [37, 38, 39],
        'UniProtKB Gene Name': [37, 38, 39],
        'Vega gene': [37, 38, 39],
        'Vega_gene': [37, 38, 39],
        'synonym_id::MGI Symbol': [37, 38, 39],
        'synonym_id::NCBI gene (formerly Entrezgene)': [37, 38, 39],
        'synonym_id::UniProtKB Gene Name': [37, 38, 39],
    },
    'transcript': {
        'CCDS': [37, 38, 39],
        'CCDS label': [37, 38, 39],
        'Havana transcript': [37, 38, 39],
        'RefSeq_mRNA': [37, 38, 39],
        'RefSeq_mRNA_predicted': [37, 38, 39],
        'RefSeq_ncRNA': [37, 38, 39],
        'RefSeq_ncRNA_predicted': [37, 38, 39],
        'EntrezGene': [37, 38, 39],
        'NCBI gene (formerly Entrezgene)': [37, 38, 39],
        'synonym_id::NCBI gene (formerly Entrezgene)': [37, 38, 39],
    },
    'translation': {
        'Havana translation': [37, 38, 39],
        'RefSeq_peptide': [37, 38, 39],
        'RefSeq_peptide_predicted': [37, 38, 39],
        'Uniprot/SPTREMBL': [37, 38, 39],
        'Uniprot/SWISSPROT': [37, 38, 39],
        'EntrezGene': [37, 38, 39],
        'NCBI gene (formerly Entrezgene)': [37, 38, 39],
        'synonym_id::NCBI gene (formerly Entrezgene)': [37, 38, 39],
    },
}

y = yaml.safe_load(template_mm.read_text(encoding='utf-8'))

# Apply the default configuration: set Include=True for matching form/database/assembly
enabled_count = 0
for form, databases in DEFAULT_MM_EXTERNALS.items():
    if form not in y[organism_mm]:
        continue
    for db_name, assemblies in databases.items():
        if db_name not in y[organism_mm][form]:
            print(f'  [SKIP] {form}/{db_name} not found in template')
            continue
        for asm_code in assemblies:
            asm_str = str(asm_code)
            if asm_str in y[organism_mm][form][db_name]['Assembly']:
                y[organism_mm][form][db_name]['Assembly'][asm_str]['Include'] = True
                enabled_count += 1

modified_mm.write_text(yaml.safe_dump(y, sort_keys=False, allow_unicode=True), encoding='utf-8')

print(f'Enabled {enabled_count} form/database/assembly combinations.')
print('Wrote:', modified_mm)


In [ ]:
_ = dm_mm.external_inst.load_modified_yaml()
print('Enabled DBs:', sorted(dm_mm.external_inst.give_list_for_case('db')))
print('Assemblies in play:', sorted(dm_mm.external_inst.give_list_for_case('assembly')))


## 2.3 — Pig (Sus scrofa)

Pig templates list multiple assemblies across history, but Ensembl is a clean-handoff species (one maintained assembly per release). Common Ensembl assembly codes you will see include:
- `111` = Sscrofa11.1
- `102` = Sscrofa10.2
- `9` = Sscrofa9

> **Tip:** For clean-handoff species, enabling the same database across the listed assemblies is usually fine;
> older assemblies mainly matter for legacy datasets and archive releases.



Pig ships with a default external YAML; regenerating it follows the same template-plus-allowlist workflow.

Pig datasets often mix different naming conventions, so enabling a small set of high-signal externals is especially important.

Commonly useful databases for pig:
- **EntrezGene**
- **Uniprot/SWISSPROT** (translation form)
- **RefSeq_mRNA** (transcript form)

> **Expected output:** You will create `sus_scrofa_externals_modified.yml` in your local repository.


In [ ]:
# 6.1 Resolve organism name + latest release
organism_ss, latest_release_ss = API.ensembl_organism('sus scrofa')
SNAPSHOT_RELEASE_SS = 115
organism_ss, SNAPSHOT_RELEASE_SS

In [ ]:
# 6.2 Create DatabaseManager snapshot
dm_ss = API.database_manager(organism_ss, SNAPSHOT_RELEASE_SS, str(LOCAL_REPOSITORY))
dm_ss

In [ ]:
%%collapse Click to show download logs
# 6.3 Generate template YAML
df_ss = dm_ss.create_database_content()
dm_ss.external_inst.create_template_yaml(df_ss)

template_ss = Path(dm_ss.external_inst.file_name_template_yaml())
modified_ss = Path(dm_ss.external_inst.file_name_modified_yaml(location='configured'))
{
    'template_yaml': str(template_ss),
    'modified_yaml': str(modified_ss),
}

### 2.3.1 Programmatic allowlist (pig)

Pig symbol databases vary across assemblies: Sscrofa10.2 carries HGNC symbols, Sscrofa11.1 VGNC symbols, so the allowlist
enables both. Use the template to inspect what is available for your snapshot release.


In [ ]:
# Default external databases for PIG.
# This dict is structured as: form -> database -> list of assemblies where Include=True

DEFAULT_SS_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [9, 102, 111],
        'Clone_based_vega_gene': [9, 102, 111],
        'EntrezGene': [9, 102, 111],
        'HGNC Symbol': [9, 102, 111],
        'VGNC Symbol': [9, 102, 111],
        'Havana gene': [9, 102, 111],
        'NCBI gene (formerly Entrezgene)': [9, 102, 111],
        'RFAM': [9, 102, 111],
        'RFAM label': [9, 102, 111],
        'miRBase': [9, 102, 111],
        'miRBase label': [9, 102, 111],
        'UniProtKB Gene Name': [9, 102, 111],
        'synonym_id::HGNC Symbol': [9, 102, 111],
        'synonym_id::NCBI gene (formerly Entrezgene)': [9, 102, 111],
        'synonym_id::UniProtKB Gene Name': [9, 102, 111],
    },
    'transcript': {
        'Havana transcript': [9, 102, 111],
        'RefSeq_mRNA': [9, 102, 111],
        'RefSeq_mRNA_predicted': [9, 102, 111],
        'RefSeq_ncRNA': [9, 102, 111],
        'RefSeq_ncRNA_predicted': [9, 102, 111],
        'EntrezGene': [9, 102, 111],
        'NCBI gene (formerly Entrezgene)': [9, 102, 111],
        'synonym_id::NCBI gene (formerly Entrezgene)': [9, 102, 111],
    },
    'translation': {
        'RefSeq_peptide': [9, 102, 111],
        'RefSeq_peptide_predicted': [9, 102, 111],
        'Uniprot/SPTREMBL': [9, 102, 111],
        'Uniprot/SWISSPROT': [9, 102, 111],
        'EntrezGene': [9, 102, 111],
        'NCBI gene (formerly Entrezgene)': [9, 102, 111],
        'synonym_id::NCBI gene (formerly Entrezgene)': [9, 102, 111],
    },
}

y = yaml.safe_load(template_ss.read_text(encoding='utf-8'))

# Apply the default configuration: set Include=True for matching form/database/assembly
enabled_count = 0
for form, databases in DEFAULT_SS_EXTERNALS.items():
    if form not in y[organism_ss]:
        continue
    for db_name, assemblies in databases.items():
        if db_name not in y[organism_ss][form]:
            print(f'  [SKIP] {form}/{db_name} not found in template')
            continue
        for asm_code in assemblies:
            asm_str = str(asm_code)
            if asm_str in y[organism_ss][form][db_name]['Assembly']:
                y[organism_ss][form][db_name]['Assembly'][asm_str]['Include'] = True
                enabled_count += 1

modified_ss.write_text(yaml.safe_dump(y, sort_keys=False, allow_unicode=True), encoding='utf-8')

print(f'Enabled {enabled_count} form/database/assembly combinations.')
print('Wrote:', modified_ss)


In [ ]:
_ = dm_ss.external_inst.load_modified_yaml()
print('Enabled DBs:', sorted(dm_ss.external_inst.give_list_for_case('db')))
print('Assemblies in play:', sorted(dm_ss.external_inst.give_list_for_case('assembly')))


## 2.4 — Rhesus macaque (Macaca mulatta)

Rhesus macaque is a clean-handoff species with three assemblies in IDTrack's window (release 48 onwards):
- `10` = Mmul_10 (releases 98+)
- `801` = Mmul_8.0.1 (releases 86–97)
- `1` = MMUL_1 (releases 48–85)

> **Note:** Ensembl names both MMUL_1 and Mmul_10 core databases `macaca_mulatta_core_<release>_10`. IDTrack gives MMUL_1 its own
> key `1` (its table entry says `DbSuffix: "10"` and `ReleaseRange: [48, 85]`), so the two genomes are never mixed; the YAML's
> assembly keys are therefore `1`, `801` and `10`.

The symbol source changes with the assembly: MMUL_1 and Mmul_8.0.1 genes carry **HGNC Symbol**, Mmul_10 genes **VGNC Symbol**
(HGNC symbols are rare there). The allowlist therefore enables both, plus the same identifier databases as for the other
organisms (EntrezGene / NCBI gene, UniProtKB gene names, RFAM, RefSeq, UniProt).

The template step reads the external tables of every release (about 70 for rhesus): it is the long step. It works over HTTPS alone;
set `IDTRACK_ENSEMBL_ACCESS=https` where the MySQL ports are closed.

> **Expected output:** `macaca_mulatta_externals_modified.yml` in your local repository.

In [ ]:
# 2.4.1 Resolve organism name + latest release ('Macaca mulatta' and 9544 work too)
organism_mmul, latest_release_mmul = API.ensembl_organism('macaca_mulatta')
SNAPSHOT_RELEASE_MMUL = 115
organism_mmul, SNAPSHOT_RELEASE_MMUL

In [ ]:
%%collapse Click to show download logs
# 2.4.2 Create a DatabaseManager snapshot and generate the template YAML (long: every release and form)
dm_mmul = API.database_manager(organism_mmul, SNAPSHOT_RELEASE_MMUL, str(LOCAL_REPOSITORY))
df_mmul = dm_mmul.create_database_content()
dm_mmul.external_inst.create_template_yaml(df_mmul)

template_mmul = Path(dm_mmul.external_inst.file_name_template_yaml())
modified_mmul = Path(dm_mmul.external_inst.file_name_modified_yaml(location='configured'))
{
    'template_yaml': str(template_mmul),
    'modified_yaml': str(modified_mmul),
}

In [ ]:
# 2.4.3 Default external databases for RHESUS MACAQUE.
# This dict is structured as: form -> database -> list of assemblies where Include=True
# Names a release does not have (e.g. CCDS, Vega) are skipped.

DEFAULT_MMUL_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [1, 801, 10],
        'Clone_based_vega_gene': [1, 801, 10],
        'EntrezGene': [1, 801, 10],
        'HGNC Symbol': [1, 801, 10],
        'VGNC Symbol': [1, 801, 10],
        'Havana gene': [1, 801, 10],
        'NCBI gene (formerly Entrezgene)': [1, 801, 10],
        'RFAM': [1, 801, 10],
        'RFAM label': [1, 801, 10],
        'miRBase': [1, 801, 10],
        'miRBase label': [1, 801, 10],
        'UniProtKB Gene Name': [1, 801, 10],
        'Vega gene': [1, 801, 10],
        'Vega_gene': [1, 801, 10],
        'synonym_id::HGNC Symbol': [1, 801, 10],
        'synonym_id::VGNC Symbol': [1, 801, 10],
        'synonym_id::NCBI gene (formerly Entrezgene)': [1, 801, 10],
        'synonym_id::UniProtKB Gene Name': [1, 801, 10],
    },
    'transcript': {
        'CCDS': [1, 801, 10],
        'CCDS label': [1, 801, 10],
        'Havana transcript': [1, 801, 10],
        'RefSeq_mRNA': [1, 801, 10],
        'RefSeq_mRNA_predicted': [1, 801, 10],
        'RefSeq_ncRNA': [1, 801, 10],
        'RefSeq_ncRNA_predicted': [1, 801, 10],
        'EntrezGene': [1, 801, 10],
        'NCBI gene (formerly Entrezgene)': [1, 801, 10],
        'synonym_id::NCBI gene (formerly Entrezgene)': [1, 801, 10],
    },
    'translation': {
        'Havana translation': [1, 801, 10],
        'RefSeq_peptide': [1, 801, 10],
        'RefSeq_peptide_predicted': [1, 801, 10],
        'Uniprot/SPTREMBL': [1, 801, 10],
        'Uniprot/SWISSPROT': [1, 801, 10],
        'EntrezGene': [1, 801, 10],
        'NCBI gene (formerly Entrezgene)': [1, 801, 10],
        'synonym_id::NCBI gene (formerly Entrezgene)': [1, 801, 10],
    },
}

from idtrack.build.default_externals import apply_allowlist
from idtrack.build.externals import write_externals_yaml

y = yaml.safe_load(template_mmul.read_text(encoding='utf-8'))
y, enabled_count, absent = apply_allowlist(y, organism_mmul, DEFAULT_MMUL_EXTERNALS)
write_externals_yaml(y, str(modified_mmul))

print(f'Enabled {enabled_count} form/database/assembly combinations; not in the template: {absent}')
print('Wrote:', modified_mmul)

In [ ]:
_ = dm_mmul.external_inst.load_modified_yaml()
print('Enabled DBs:', sorted(dm_mmul.external_inst.give_list_for_case('db')))
print('Assemblies in play:', sorted(dm_mmul.external_inst.give_list_for_case('assembly')))

## 2.5 — Cynomolgus macaque (Macaca fascicularis)

Cynomolgus (crab-eating) macaque has two assemblies, with a clean handoff at release 103:
- `6` = Macaca_fascicularis_6.0 (releases 103+)
- `5` = Macaca_fascicularis_5.0 (releases 91–102)

As for rhesus, 5.0 genes carry **HGNC Symbol** and 6.0 genes **VGNC Symbol**, but symbol coverage of 6.0 in Ensembl is thin (a few
hundred genes of each kind); EntrezGene / NCBI gene and UniProtKB gene names cover most genes. Ensembl lists cynomolgus under the
common name "Crab-eating macaque".

> **Expected output:** `macaca_fascicularis_externals_modified.yml` in your local repository.

In [ ]:
# 2.5.1 Resolve organism name + latest release ('Crab-eating macaque' and 9541 work too)
organism_mfas, latest_release_mfas = API.ensembl_organism('macaca_fascicularis')
SNAPSHOT_RELEASE_MFAS = 115
organism_mfas, SNAPSHOT_RELEASE_MFAS

In [ ]:
%%collapse Click to show download logs
# 2.5.2 Create a DatabaseManager snapshot and generate the template YAML (long: every release and form)
dm_mfas = API.database_manager(organism_mfas, SNAPSHOT_RELEASE_MFAS, str(LOCAL_REPOSITORY))
df_mfas = dm_mfas.create_database_content()
dm_mfas.external_inst.create_template_yaml(df_mfas)

template_mfas = Path(dm_mfas.external_inst.file_name_template_yaml())
modified_mfas = Path(dm_mfas.external_inst.file_name_modified_yaml(location='configured'))
{
    'template_yaml': str(template_mfas),
    'modified_yaml': str(modified_mfas),
}

In [ ]:
# 2.5.3 Default external databases for CYNOMOLGUS MACAQUE.
# This dict is structured as: form -> database -> list of assemblies where Include=True

DEFAULT_MFAS_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [5, 6],
        'Clone_based_vega_gene': [5, 6],
        'EntrezGene': [5, 6],
        'HGNC Symbol': [5, 6],
        'VGNC Symbol': [5, 6],
        'Havana gene': [5, 6],
        'NCBI gene (formerly Entrezgene)': [5, 6],
        'RFAM': [5, 6],
        'RFAM label': [5, 6],
        'miRBase': [5, 6],
        'miRBase label': [5, 6],
        'UniProtKB Gene Name': [5, 6],
        'Vega gene': [5, 6],
        'Vega_gene': [5, 6],
        'synonym_id::HGNC Symbol': [5, 6],
        'synonym_id::VGNC Symbol': [5, 6],
        'synonym_id::NCBI gene (formerly Entrezgene)': [5, 6],
        'synonym_id::UniProtKB Gene Name': [5, 6],
    },
    'transcript': {
        'CCDS': [5, 6],
        'CCDS label': [5, 6],
        'Havana transcript': [5, 6],
        'RefSeq_mRNA': [5, 6],
        'RefSeq_mRNA_predicted': [5, 6],
        'RefSeq_ncRNA': [5, 6],
        'RefSeq_ncRNA_predicted': [5, 6],
        'EntrezGene': [5, 6],
        'NCBI gene (formerly Entrezgene)': [5, 6],
        'synonym_id::NCBI gene (formerly Entrezgene)': [5, 6],
    },
    'translation': {
        'Havana translation': [5, 6],
        'RefSeq_peptide': [5, 6],
        'RefSeq_peptide_predicted': [5, 6],
        'Uniprot/SPTREMBL': [5, 6],
        'Uniprot/SWISSPROT': [5, 6],
        'EntrezGene': [5, 6],
        'NCBI gene (formerly Entrezgene)': [5, 6],
        'synonym_id::NCBI gene (formerly Entrezgene)': [5, 6],
    },
}

y = yaml.safe_load(template_mfas.read_text(encoding='utf-8'))
y, enabled_count, absent = apply_allowlist(y, organism_mfas, DEFAULT_MFAS_EXTERNALS)
write_externals_yaml(y, str(modified_mfas))

print(f'Enabled {enabled_count} form/database/assembly combinations; not in the template: {absent}')
print('Wrote:', modified_mfas)

In [ ]:
_ = dm_mfas.external_inst.load_modified_yaml()
print('Enabled DBs:', sorted(dm_mfas.external_inst.give_list_for_case('db')))
print('Assemblies in play:', sorted(dm_mfas.external_inst.give_list_for_case('assembly')))

## 2.6 — Extending a YAML to a new Ensembl release

Each Ensembl release can add or drop external databases, and a YAML file lists, per database and assembly, the releases it was
made from. A graph whose snapshot includes a release that the YAML does not list gets **no external identifiers for that release**;
IDTrack warns about it once when it builds the graph.

You do not need to regenerate and re-curate the whole file. `ExternalDatabases.extend_modified_yaml(content)` takes the
database-content table of the new release(s), keeps a copy of the file it replaces (`<file>.bak`), and:
- adds the new release(s) to every entry the table holds, keeping every `Include` flag as you set it;
- adds databases the file does not know yet with `Include: false` and lists them, so you can decide whether to enable them;
- adds a known database that appears with a new assembly, with the database's flag when its other assemblies agree.

The content table comes from a manager whose window is just the new release(s), which keeps the download to one release per
assembly. The extended file is written to your local repository (or to `output_path=`).

In [ ]:
%%collapse Click to show download logs
# 2.6.1 Extend the human YAML to a new release (here: Ensembl's newest one)
NEW_RELEASE = latest_release_hs

dm_new = idtrack.build.DatabaseManager(
    organism='homo_sapiens',
    form='gene',
    local_repository=str(LOCAL_REPOSITORY),
    first_release=NEW_RELEASE,
    snapshot_release=NEW_RELEASE,
)
content_new = dm_new.create_database_content()
report = dm_new.external_inst.extend_modified_yaml(content_new)

print('Written:', report['path'])
print('Releases added:', report['releases'], '| entries extended:', report['extended_entries'])
print('New databases (Include: false, review them):', report['new_databases'])
print('New assembly entries of known databases:', report['new_assembly_entries'])

## 2.7 — Adding a New Organism (Advanced)

IDTrack ships with built-in support for human, mouse, pig, rhesus macaque and cynomolgus macaque. Another Ensembl species needs
**two things**:

1. **An entry in the organism table** (`ORGANISM_TABLE` in `idtrack/organisms.py`): one entry per assembly, keyed by an
   integer, with the MySQL `Ports`, a `Priority` (1 = preferred), the `ReleaseRange` of the assembly and, optionally, its Ensembl
   `Name` and `DbSuffix` (the code in core database names `<organism>_core_<release>_<code>`, when it differs from the key). To try
   an organism without editing the package, register it for the running session with `idtrack.organisms.register_organism(name, assemblies,
   symbol_databases, aliases)`; it validates the entry and updates everything derived from the table.
   - Direct MySQL connectivity is optional: when ports are blocked, IDTrack downloads the same tables from the HTTPS/FTP MySQL dumps.
2. **An externals YAML**: generate a template and apply an allowlist exactly as for the organisms above.

IDTrack reads Ensembl ids with their versions: an organism whose Ensembl ids carry no versions is not supported (a graph build refuses
it with an error that says so).

Practical recipe:

- Step A: Resolve the canonical Ensembl species name (snake_case) with `idtrack.API.ensembl_organism(...)`.
- Step B: Find its assemblies in the dump listings (`https://ftp.ensembl.org/pub/release-<n>/mysql/`) and register them (example
  below). Listing all assemblies is what enables cross-assembly mapping and assembly-scoped external databases.
- Step C: Run the YAML-template generation workflow (`DatabaseManager.create_database_content(...)` → `create_template_yaml(...)`)
  and apply an allowlist.
- Step D: Build a graph snapshot (Part 3) and run the sanity checks.

The full recipe (including the tests and docs to update when adding an organism to IDTrack itself) is "Adding Organism Support" in
the contributing guide.

> **Warning:** Until Step B is done, `DatabaseManager` raises `NotImplementedError` for that organism.

In [ ]:
# Minimal helper cell: check if an organism is supported by *this* IDTrack version.
# (Safe to run; it will not modify your installation.)

from idtrack import organisms

print('Supported organisms in this IDTrack version:')
print('  ' + ', '.join(organisms.supported_organisms()))

# Example: resolve an organism name via Ensembl REST (works even if the organism isn't configured locally)
organism_query = 'danio rerio'  # zebrafish (example)
formal_name, latest_release = API.ensembl_organism(organism_query)
print('Resolved via Ensembl REST ->', organism_query, '→', formal_name, '(latest release:', latest_release, ')')

if formal_name not in organisms.supported_organisms():
    print()
    print('This organism is not yet configured in this IDTrack version.')
    print('To add it for this session: organisms.register_organism(name, {key: {...}}, symbol_databases) (example below);')
    print('then generate its external YAML with the same workflow as the organisms above.')
else:
    print()
    print('This organism is already configured. You can now generate a template YAML and continue.')

In [ ]:
# Example: register zebrafish (GRCz11 core databases are named danio_rerio_core_<release>_11) for this Python session.
# Check the release range against the dump listings before relying on it.
organisms.register_organism(
    'danio_rerio',
    {11: {'Name': 'GRCz11', 'Ports': [3306, 5306], 'Priority': 1, 'ReleaseRange': [92, None]}},
    symbol_databases=['ZFIN'],
)
print('danio_rerio' in organisms.supported_organisms(), sorted(organisms.ORGANISM_TABLE['danio_rerio']))

## 2.8 — Final checklist (before you build graphs)

You should now have these files in your local repository (or rely on the shipped defaults for the organisms you skipped):
- `homo_sapiens_externals_modified.yml`
- `mus_musculus_externals_modified.yml`
- `sus_scrofa_externals_modified.yml`
- `macaca_mulatta_externals_modified.yml`
- `macaca_fascicularis_externals_modified.yml`

The next notebook (`03_initialization_graph.ipynb`) will build graphs using these configs.


In [ ]:
# Quick existence check
expected = [
    LOCAL_REPOSITORY / 'homo_sapiens_externals_modified.yml',
    LOCAL_REPOSITORY / 'mus_musculus_externals_modified.yml',
    LOCAL_REPOSITORY / 'sus_scrofa_externals_modified.yml',
    LOCAL_REPOSITORY / 'macaca_mulatta_externals_modified.yml',
    LOCAL_REPOSITORY / 'macaca_fascicularis_externals_modified.yml',
]
for p in expected:
    print('OK' if p.exists() else 'MISSING', '-', p)


## 2.9 — Best practices (once you get comfortable)

1. **Keep allowlists small**: enabling many overlapping databases often increases ambiguity.
2. **Multiple profiles**: you can maintain different YAMLs per project (e.g. ‘strict’ vs ‘broad’), one local repository per profile
   (a graph file records the selection it was built from; see Part 7).
3. **Assembly awareness**:
   - In Ensembl schema names, the last number encodes the genome assembly (e.g. human 38 = GRCh38; mouse 39 = GRCm39; pig 111 = Sscrofa11.1).
     IDTrack's assembly keys are these codes, except rhesus MMUL_1 (key `1`), whose databases share the code `10` with Mmul_10.
   - Assembly codes are species-specific, and external databases can be assembly-scoped.
   - Keeping all assembly blocks in your YAML is usually beneficial when you integrate mixed-build datasets.
4. **New Ensembl releases**: extend your YAML with `extend_modified_yaml` (section 2.6) instead of regenerating it; review the
   databases it reports as new.
